<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Pixal3D_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pixal3D — Pixel-Aligned 3D Generation (SIGGRAPH 2026)

State-of-the-art image-to-3D by Tencent ARC Lab. Generates high-fidelity GLB meshes with PBR textures from a single image. Built on Microsoft's TRELLIS.2 backbone.

### Quick Start
1. Connect to a GPU runtime (A100, L4, or T4)
2. Run all steps in order — no token, no sign-up needed
3. Launch the Gradio UI (Step 4) or test with a single image (Step 6)

| GPU | VRAM | Resolution | Mode |
|-----|------|-----------|------|
| A100 | 40 GB | 1536 px | Standard |
| L4 | 24 GB | 1024 px | Low-VRAM (recommended) |
| T4 | 15 GB | 1024 px | Low-VRAM (auto) |

Upload images to `/content/images_in/`, then run **Step 7** to batch-process.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs prebuilt CUDA wheels compiled for Colab GPUs from GitHub Releases.

import os, sys, subprocess
import torch
print("=" * 72)
print("Pixal3D STEP 1 — Install Dependencies")
print("=" * 72)
print(f"  Python : {sys.version.split()[0]}")
print(f"  torch  : {torch.__version__}  CUDA: {torch.version.cuda}")


if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# Auto-detect GPU
gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
arch = f"{cap[0]}{cap[1]}"
# Map SM arch to release tag suffix
arch_tag = {'75': 't4', '80': 'a100', '89': 'l4'}.get(arch, None)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'[GPU] Detected {gpu_name} ({vram_gb:.0f} GB) \u2192 sm_{arch} ({arch_tag})')
if arch_tag is None:
    raise SystemExit(
        f'[ERROR] No Pixal3D CUDA wheels are available for sm_{arch} ({gpu_name}).  '
        'Pixal3D prebuilt wheels cover T4 (sm_75), A100 (sm_80), and L4 (sm_89).  '
        'H100 (sm_90) and Blackwell (sm_100) require building the wheels yourself:  '
        'open Pixal3D_Wheel_Builder.ipynb, set gpu_tag to the right value, run all cells,  '
        'and upload the new wheels to a matching GitHub Release tag.'
    )
os.makedirs('/content/images_in', exist_ok=True)

# Drive-cached HuggingFace weights (3-5 GB; survives Colab session resets).
# Set BEFORE any huggingface_hub import (snapshot_download in STEP 2).
os.environ.setdefault('HF_HOME', '/content/drive/MyDrive/AEI_3D_Cache/huggingface')
os.makedirs(os.environ['HF_HOME'], exist_ok=True)


# Pin torch to 2.11.0+cu128 -- the version the prebuilt L4 wheels were
# compiled against (build date 2026-05-29). Colab auto-upgrades torch
# to newer versions over time; a C++ extension built for 2.11 has a
# different c10::IValue arg layout than 2.12+, so kernel launches from
# cumesh / flex_gemm / o_voxel silently fail with "no kernel image"
# at inference time even though __import__() succeeds at install time.
# Pin to 2.11.0+cu128 to match the wheels.  This is a NO-OP if the
# runtime already has 2.11.x.  When the pin fires, the kernel
# auto-restarts via os.execv; re-running the cell the second time
# then sees 2.11.0 and continues to the wheel install.
_target_torch = "2.11.0"
if not torch.__version__.startswith(_target_torch):
    print(f"[Fix] Colab has torch {torch.__version__}; reinstalling {_target_torch}.0+cu128 ...")
    subprocess.run([
        "pip", "install", "-q", "--disable-pip-version-check", "--no-input",
        f"torch=={_target_torch}.0+cu128",
        "torchvision==0.26.0+cu128",
        "torchaudio==2.11.0+cu128",
        "--index-url", "https://download.pytorch.org/whl/cu128",
        "--force-reinstall",
    ], check=False)
    # Force a kernel restart via os.execv so the new torch takes effect.
    # The restarted process re-runs this cell from the top, sees the
    # already-correct 2.11.x torch, and continues to wheel install.
    print("[Fix] Restarting kernel to load new torch 2.11.0+cu128 ...")
    import os as _os
    _os.execv(sys.executable, [sys.executable] + sys.argv)


# Prebuilt CUDA wheels from GitHub Releases
# Wheels rebuilt on 2026-07-01 against the latest Colab runtime
# (torch 2.11.0+cu128, pydantic 2.11.10, CUDA 12.8, libcu12 bundled).
# v1.0 was built on 2026-05-29 against an earlier runtime and now
# has C++ ABI drift; v1.1 replaces it.  After this notebook installs
# v1.1 wheels, future runs land on v1.1 by default (cached via the
# pip install line below).
WHEELS_TAG = f'wheels-{arch_tag}-v1.1'
RELEASE = f'https://github.com/Skquark/AEI-Colab-Notebooks/releases/download/{WHEELS_TAG}'

# Exact wheel filenames (same names across all GPU archs)
WHEELS = {
    'o_voxel': 'o_voxel-0.0.1-cp312-cp312-linux_x86_64.whl',
    'cumesh': 'cumesh-0.0.1-cp312-cp312-linux_x86_64.whl',
    'flex_gemm': 'flex_gemm-1.0.0-cp312-cp312-linux_x86_64.whl',
    'nvdiffrec_render': 'nvdiffrec_render-0.0.0-cp312-cp312-linux_x86_64.whl',
}

for name, whl in WHEELS.items():
    url = f'{RELEASE}/{whl}'
    print(f'  Installing {name}...')
    try:
        subprocess.run(['pip', 'install', '-q', '--no-deps', url], check=True)
    except Exception as e:
        print(f'  [WARN] {name}: {e}')
print('[CUDA Packages] Installed')

# Install trimesh and a few small Python deps BEFORE the CUDA-wheel
# __import__() verification.  o_voxel imports trimesh at module load
# time, so the wheel verify fails if trimesh is not yet installed.
# Suppress the noisy pip-dependency-resolver warnings from Colab
# pre-installed packages.  Colab pre-installs google-adk 2.3.0,
# google-genai 2.10.0, hf-gradio, python-fasthtml, etc., and these all
# pin modern pydantic>=2.12 / starlette>=1.0 / gradio-client>=2.0.
# None of them are Pixal3D deps; we just bump them so the resolver
# doesn't spam warnings on every pip install.  These bumps are
# idempotent -- re-running STEP 1 always lands on a known-good state.
_pip_quiet = ["-q", "--disable-pip-version-check", "--no-input"]
try:
    import pydantic as _pyd_check
    if not _pyd_check.VERSION.startswith(("2.12", "2.13")):
        print(f"[Fix] Colab has pydantic {_pyd_check.VERSION}; bumping to >=2.12,<3 ...")
        subprocess.run(["pip", "install"] + _pip_quiet + ["pydantic>=2.12,<3"], check=False)
except ImportError:
    pass
try:
    import starlette as _stl_check
    # google-adk 2.3.0 / python-fasthtml 0.14+ require starlette>=1.0.1.
    major = int(_stl_check.__version__.split(".")[0])
    if major < 1:
        print(f"[Fix] Colab has starlette {_stl_check.__version__}; bumping to >=1.0.1,<2 ...")
        subprocess.run(["pip", "install"] + _pip_quiet + ["starlette>=1.0.1,<2"], check=False)
except ImportError:
    pass
try:
    import gradio_client as _gc_check
    # hf-gradio 0.4.1 requires gradio-client>=2.0; the pre-installed Colab
    # version is 1.x.  We let STEP 4 install the right gradio which
    # drags in the matching client, so just log here.
    major = int(_gc_check.__version__.split(".")[0])
    if major < 2:
        print(f"[Fix] Colab has gradio-client {_gc_check.__version__}; will be upgraded by gradio==5.49.1 below")
except ImportError:
    pass

subprocess.run(["pip", "install", "-q", "trimesh", "pygltflib", "plyfile",
                  "easydict", "huggingface_hub", "tqdm"], check=False)

# nvdiffrast (compile from NVlabs source \u2014 links against local torch)
print('[nvdiffrast] Compiling from NVlabs source...')
!pip install -q ninja
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
subprocess.run(['pip', 'install', '--no-build-isolation', 'git+https://github.com/NVlabs/nvdiffrast.git'], check=True)
print('[nvdiffrast] Done')


# Verify CUDA packages imported correctly
import traceback
for name, display in [('o_voxel', 'o_voxel'), ('cumesh', 'cumesh'),
                      ('flex_gemm', 'flex_gemm'), ('nvdiffrec_render', 'nvdiffrec_render')]:
    try:
        __import__(name)
        print(f'  Verified: {display}')
    except Exception as _e:
        print(f'  [FAIL] {display}')
        traceback.print_exc()
        print()

# NOTE: transformers is pinned to 4.57.3 because that is the version
# Pixal3D was tested against.  In transformers 5.x the DINOv3ViTModel
# wraps the encoder in `self.model.model` (renamed from `self.model`),
# which breaks Pixal3D's `pixal3d/modules/image_feature_extractor.py`
# (it does `for layer_module in self.model.layer:` and fails with
# AttributeError when the new transformers ships an extra Encoder wrapper).
import transformers as _tf_check
if not _tf_check.__version__.startswith("4.57"):
    print(f"[Fix] Colab has transformers {_tf_check.__version__}; pinning to 4.57.3 ...")
    subprocess.run(["pip", "install", "-q", "transformers==4.57.3"], check=True)
    import importlib, sys
    for m in list(sys.modules):
        if m.startswith("transformers"):
            del sys.modules[m]
    import transformers
    print(f"[Fix] transformers now {transformers.__version__}")

# Python dependencies
!pip install -q trimesh pygltflib plyfile moderngl huggingface_hub kornia kornia-rs \
    imageio imageio-ffmpeg tqdm easydict opencv-python-headless \
    zstandard timm diffusers accelerate gradio==5.49.1 einops onnxruntime
!pip install -q git+https://github.com/microsoft/MoGe.git

# natten (prebuilt wheel for NAF upsampling)
natten_ok = False
try:
    print(f'[natten] Installing for sm_{arch}...')
    !pip install natten==0.21.6+torch2110cu128 -f https://whl.natten.org -q
    import natten; natten_ok = getattr(natten, 'HAS_LIBNATTEN', False)
    print('  natten - done' if natten_ok else '  natten - no libnatten, NAF disabled')
except Exception:
    print('  natten - skipped (NAF disabled)')

!pip install -q --force-reinstall --no-deps \
    https://github.com/LDYang694/Storages/releases/download/20260430/utils3d-0.0.2-py3-none-any.whl


# Re-run the Colab-dep pin checks at the end.  MoGe, nvdiffrast, or
# any other `pip install` above may have downgraded pydantic/starlette
# to satisfy their own pins.  Re-bump here so the resolver stays quiet.
try:
    import pydantic as _pyd_re
    if not _pyd_re.VERSION.startswith(("2.12", "2.13")):
        print(f"[Fix] Late check: pydantic is {_pyd_re.VERSION}; re-bumping to >=2.12,<3")
        subprocess.run(["pip", "install"] + _pip_quiet + ["pydantic>=2.12,<3"], check=False)
except ImportError:
    pass
try:
    import starlette as _stl_re
    major = int(_stl_re.__version__.split(".")[0])
    if major < 1:
        print(f"[Fix] Late check: starlette is {_stl_re.__version__}; re-bumping to >=1.0.1,<2")
        subprocess.run(["pip", "install"] + _pip_quiet + ["starlette>=1.0.1,<2"], check=False)
except ImportError:
    pass

print(f'\n[DONE] {gpu_name} ({vram_gb:.0f} GB) \u2014 all dependencies installed')


In [ ]:
# @title STEP 2 — Clone Repo & Pre-cache Models
# @markdown Clones Pixal3D and pre-downloads model weights from HuggingFace (~3-5 GB). Run once — cached for future sessions.
import os, subprocess

os.chdir('/content')
if not os.path.exists('/content/Pixal3D/.git'):
    if os.path.exists('/content/Pixal3D'):
        import shutil; shutil.rmtree('/content/Pixal3D')
    subprocess.run(['git', 'clone', 'https://github.com/TencentARC/Pixal3D.git', '/content/Pixal3D'], check=True)
else:
    print('[Step 2] Pixal3D already cloned')

os.chdir('/content/Pixal3D')
subprocess.run(['git', 'lfs', 'pull'], capture_output=True)

from huggingface_hub import snapshot_download

print('[Cache] DinoV3 feature extractor...')
snapshot_download('camenduru/dinov3-vitl16-pretrain-lvd1689m',
    allow_patterns=['*.bin','*.json','*.txt','*.model','*.safetensors'], max_workers=4)

print('[Cache] BiRefNet (background removal)...')
snapshot_download('ZhengPeng7/BiRefNet',
    allow_patterns=['*.pth','*.json','*.txt','*.bin','*.model'], max_workers=2)

hdri_dir = '/content/Pixal3D/assets/hdri'
if os.path.isdir(hdri_dir):
    exr_files = [f for f in os.listdir(hdri_dir) if f.endswith('.exr')]
    if exr_files: print(f'[Assets] {len(exr_files)} HDRI maps found')
    else: print('[Warning] No .exr files — preview renders use fallback')

print('\n[DONE] All models cached')
os.makedirs('/content/Pixal3D/output', exist_ok=True)

In [ ]:
# @title STEP 3 — Environment & Core Functions
# @markdown Sets environment variables, attention backends, and defines `init_models()` and `run_inference()`.
DECIMATION = 700000 # @param {type:"integer"}
TEXTURE_SIZE = 4096 # @param [2048, 4096] {type:"raw"}

import os, sys, math, time, json, glob, gc
import torch, numpy as np
from PIL import Image
from datetime import datetime
# Suppress Python 3.12+ SyntaxWarnings raised by the upstream
# Pixal3D source (e.g. "/mathcal/E" in docstrings).  These fire every
# time the pipeline is re-imported inside a fresh subprocess or worker,
# which spams the Step 7 batch output -- see warning text below.
# We target ONLY the SyntaxWarning category (other warnings stay loud).
import warnings
warnings.filterwarnings(
    'ignore', category=SyntaxWarning,
    message=r'.*invalid escape sequence.*',
)


os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# CUDA_LAUNCH_BLOCKING=1 makes CUDA errors synchronous so we get a real
# stack trace instead of the 'might be asynchronously reported' fallback.
# Keep this set; comment out only if the synchronous errors slow things down too much.
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Attention backend auto-detect
attn_backend = 'sdpa'
try:
    import flash_attn_interface; attn_backend = 'flash_attn_3'
    print('[Attn] FlashAttention 3')
except ImportError:
    try:
        import flash_attn; attn_backend = 'flash_attn'
        print('[Attn] FlashAttention 2')
    except ImportError:
        print('[Attn] SDPA fallback')
os.environ['ATTN_BACKEND'] = attn_backend
os.environ['SPARSE_ATTN_BACKEND'] = attn_backend
os.environ['FLEX_GEMM_AUTOTUNE_CACHE_PATH'] = '/content/autotune_cache.json'
os.environ['FLEX_GEMM_AUTOTUNER_VERBOSE'] = '1'

import cv2

os.chdir('/content/Pixal3D')
if '/content/Pixal3D' not in sys.path:
    sys.path.insert(0, '/content/Pixal3D')

# Config
MODEL_PATH = 'TencentARC/Pixal3D'
MOGE_MODEL_NAME = 'Ruicheng/moge-2-vitl'

IMAGE_COND_CONFIGS = {
    'ss': {'model_name': 'camenduru/dinov3-vitl16-pretrain-lvd1689m', 'image_size': 512,  'grid_resolution': 16},
    'shape_512':  {'model_name': 'camenduru/dinov3-vitl16-pretrain-lvd1689m', 'image_size': 512,  'grid_resolution': 32,  'use_naf_upsample': True, 'naf_target_size': 512},
    'shape_1024': {'model_name': 'camenduru/dinov3-vitl16-pretrain-lvd1689m', 'image_size': 1024, 'grid_resolution': 64,  'use_naf_upsample': True, 'naf_target_size': 512},
    'tex_1024':   {'model_name': 'camenduru/dinov3-vitl16-pretrain-lvd1689m', 'image_size': 1024, 'grid_resolution': 64,  'use_naf_upsample': True, 'naf_target_size': 1024},
}

if not globals().get('natten_ok', False):
    print('[NAF] NATTEN unavailable — disabling NAF upsampling')
    for k in ('shape_512', 'shape_1024', 'tex_1024'):
        IMAGE_COND_CONFIGS[k] = {**IMAGE_COND_CONFIGS[k], 'use_naf_upsample': False}

pipeline = None
moge_model = None
envmap = None
_low_vram_active = None

def build_image_cond_model(config):
    from pixal3d.trainers.flow_matching.mixins.image_conditioned_proj import DinoV3ProjFeatureExtractor
    m = DinoV3ProjFeatureExtractor(**config); m.eval(); return m

def load_moge_model(device='cuda'):
    from moge.model.v2 import MoGeModel
    m = MoGeModel.from_pretrained(MOGE_MODEL_NAME).to(device); m.eval(); return m

def cleanup_memory():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()

def init_models(low_vram=False):
    global pipeline, moge_model, envmap, _low_vram_active
    if pipeline is not None and _low_vram_active == low_vram:
        return
    _low_vram_active = low_vram
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_name = torch.cuda.get_device_name(0)
    print(f"{'='*60}")
    print(f'[Init] {gpu_name} ({vram:.0f} GB) | Low-VRAM: {low_vram}')
    print(f"{'='*60}")

    from pixal3d.pipelines import Pixal3DImageTo3DPipeline
    print(f'[Pipeline] Loading {MODEL_PATH}...')
    pipeline = Pixal3DImageTo3DPipeline.from_pretrained(MODEL_PATH)

    print('[ImageCond] Building DinoV3 models...')
    pipeline.image_cond_model_ss = build_image_cond_model(IMAGE_COND_CONFIGS['ss'])
    pipeline.image_cond_model_shape_512 = build_image_cond_model(IMAGE_COND_CONFIGS['shape_512'])
    pipeline.image_cond_model_shape_1024 = build_image_cond_model(IMAGE_COND_CONFIGS['shape_1024'])
    pipeline.image_cond_model_tex_1024 = build_image_cond_model(IMAGE_COND_CONFIGS['tex_1024'])

    if low_vram:
        print('[NAF] Pre-downloading weights (CPU)...')
        for a in ['image_cond_model_ss','image_cond_model_shape_512','image_cond_model_shape_1024','image_cond_model_tex_1024']:
            m = getattr(pipeline, a, None)
            if m is not None and getattr(m, 'use_naf_upsample', False):
                m._load_naf()
        pipeline._device = torch.device('cuda')
        pipeline.low_vram = True
    else:
        pipeline.low_vram = False
        pipeline.cuda()
        pipeline.image_cond_model_ss.cuda()
        pipeline.image_cond_model_shape_512.cuda()
        pipeline.image_cond_model_shape_1024.cuda()
        pipeline.image_cond_model_tex_1024.cuda()
        print('[NAF] Pre-loading upsampler...')
        for a in ['image_cond_model_ss','image_cond_model_shape_512','image_cond_model_shape_1024','image_cond_model_tex_1024']:
            m = getattr(pipeline, a, None)
            if m is not None and getattr(m, 'use_naf_upsample', False):
                m._load_naf()

    print('[MoGe-2] Loading camera model...')
    moge_model = load_moge_model(device='cpu' if low_vram else 'cuda')

    print('[EnvMap] Loading HDRI...')
    from pixal3d.renderers import EnvMap
    dev = 'cpu' if low_vram else 'cuda'
    envmap = {}
    _base = '/content/Pixal3D/assets/hdri'
    for name in ['forest','sunset','courtyard']:
        p = os.path.join(_base, f'{name}.exr')
        if os.path.exists(p):
            envmap[name] = EnvMap(torch.tensor(cv2.cvtColor(cv2.imread(p, cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB), dtype=torch.float32, device=dev))
    print(f'[Init] {len(envmap)} HDRI maps loaded. Ready!')

def compute_f_pixels(camera_angle_x, resolution):
    fl = 16.0 / torch.tan(torch.tensor(camera_angle_x / 2.0))
    return float((fl * resolution / 32.0).item())

def distance_from_fov(camera_angle_x, grid_point, target_point, mesh_scale, image_resolution):
    R = torch.tensor([[1.,0.,0.],[0.,0.,-1.],[0.,1.,0.]])
    gp = grid_point.float() @ R.T / mesh_scale / 2
    xt, yt = float(target_point[0].item()), float(target_point[1].item())
    fp = compute_f_pixels(camera_angle_x, image_resolution)
    return {'distance_from_x': float(fp * gp[0].item() / (xt - image_resolution/2) - gp[1].item()), 'f_pixels': fp}

def get_camera_params(image_path, moge, device='cuda', mesh_scale=1.0, extend_pixel=0, image_resolution=512):
    global _low_vram_active
    if _low_vram_active: moge.to(device)
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    t = torch.from_numpy(np.array(img).astype(np.float32)/255.).permute(2,0,1).to(device)
    with torch.no_grad(): out = moge.infer(t)
    if _low_vram_active: moge.cpu(); cleanup_memory()
    fx_norm = out['intrinsics'].squeeze().cpu().numpy()[0,0]
    camera_angle_x = 2 * math.atan(w / (2 * fx_norm * w))
    d = distance_from_fov(camera_angle_x, torch.tensor([-1.,0.,0.]),
        torch.tensor([0-extend_pixel, image_resolution-1+extend_pixel]), mesh_scale, image_resolution)
    return {'camera_angle_x': camera_angle_x, 'distance': d['distance_from_x'], 'mesh_scale': mesh_scale}

def run_inference(image_path, seed=42, resolution=1536, low_vram=False,
                  ss_steps=12, ss_guidance=7.5, ss_rescale=0.7, ss_rescale_t=5.0,
                  shape_steps=12, shape_guidance=7.5, shape_rescale=0.5, shape_rescale_t=3.0,
                  tex_steps=12, tex_guidance=1.0, tex_rescale=0.0, tex_rescale_t=3.0,
        manual_fov=-1.0, fov_unit='deg', render_mode='shaded_forest',
                  output_path=None, render_preview=False, verbose=True):
    global pipeline, moge_model, envmap
    t0 = time.time()
    if pipeline is None: init_models(low_vram=low_vram)
    if verbose: print(f'[Inference] {os.path.basename(image_path)}')
    img = Image.open(image_path).convert('RGBA')
    image_preprocessed = pipeline.preprocess_image(img)
    if manual_fov > 0:
        if fov_unit == 'rad':
            camera_angle_x = float(manual_fov)
        else:
            camera_angle_x = math.radians(manual_fov)
        d = distance_from_fov(camera_angle_x, torch.tensor([-1.,0.,0.]),
            torch.tensor([0, 511]), 1.0, 512)
        cam = {'camera_angle_x': camera_angle_x, 'distance': d['distance_from_x'], 'mesh_scale': 1.0}
        if verbose: print(f"  Camera (manual): fov={math.degrees(camera_angle_x):.1f}°, dist={cam['distance']:.3f}")
    else:
        tmp = '/content/_tmp_cam.png'; image_preprocessed.save(tmp)
        cam = get_camera_params(tmp, moge_model, mesh_scale=1.0, extend_pixel=0, image_resolution=512)
        os.remove(tmp)
        if verbose: print(f"  Camera (auto): fov={math.degrees(cam['camera_angle_x']):.1f}°, dist={cam['distance']:.3f}")
    torch.manual_seed(int(seed))
    hr_res = int(resolution)
    ss = {'steps': int(ss_steps), 'guidance_strength': float(ss_guidance),
          'guidance_rescale': float(ss_rescale), 'rescale_t': float(ss_rescale_t)}
    sh = {'steps': int(shape_steps), 'guidance_strength': float(shape_guidance),
          'guidance_rescale': float(shape_rescale), 'rescale_t': float(shape_rescale_t)}
    tx = {'steps': int(tex_steps), 'guidance_strength': float(tex_guidance),
          'guidance_rescale': float(tex_rescale), 'rescale_t': float(tex_rescale_t)}
    pipe_type = f'{hr_res}_cascade'
    if verbose: print(f'  Pipeline: {pipe_type}')
    mesh_list, (shape_slat, tex_slat, res) = pipeline.run(
        image_preprocessed, camera_params=cam, seed=int(seed),
        sparse_structure_sampler_params=ss, shape_slat_sampler_params=sh,
        tex_slat_sampler_params=tx, preprocess_image=False, return_latent=True,
        pipeline_type=pipe_type, max_num_tokens=49152)
    import o_voxel
    mesh = mesh_list[0]
    glb = o_voxel.postprocess.to_glb(
        vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs,
        coords=mesh.coords, attr_layout=pipeline.pbr_attr_layout,
        grid_size=res, aabb=[[-0.5,-0.5,-0.5],[0.5,0.5,0.5]],
        decimation_target=DECIMATION, texture_size=TEXTURE_SIZE,
        remesh=True, remesh_band=1, remesh_project=0, use_tqdm=True)
    rot = np.array([[-1,0,0,0],[0,0,-1,0],[0,-1,0,0],[0,0,0,1]], dtype=np.float64)
    glb.apply_transform(rot)
    if output_path is None:
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        name = os.path.splitext(os.path.basename(image_path))[0]
        output_path = f'/content/Pixal3D/output/{name}_{ts}.glb'
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    glb.export(output_path, extension_webp=True)
    preview_path = None
    if render_preview:
        try:
            from pixal3d.utils import render_utils
            from pixal3d.renderers import EnvMap
            cd = cam['distance']
            if envmap:
                if low_vram:
                    for v in envmap.values():
                        v.image = v.image.cuda()
                        if hasattr(v, '_nvdiffrec_envlight'): del v._nvdiffrec_envlight
                mesh.simplify(16777216)
                renders = render_utils.render_proj_aligned_video(
                    mesh, camera_angle_x=cam['camera_angle_x'], distance=cd,
                    resolution=512, num_frames=8, envmap=envmap,
                    near=max(0.01, cd-2.0), far=cd+10.0)
                if low_vram:
                    for v in envmap.values():
                        if hasattr(v, '_nvdiffrec_envlight'): del v._nvdiffrec_envlight
                        v.image = v.image.cpu()
                    torch.cuda.empty_cache()
                k = render_mode if render_mode in renders else list(renders.keys())[0]
                preview_path = output_path.replace('.glb','_preview.png')
                Image.fromarray(renders[k][len(renders[k])//2]).save(preview_path)
        except Exception as e:
            if verbose: print(f'  [Preview skip] {e}')
    # Clear VRAM used by mesh/renders. Keeps pipeline loaded for next call.
    del mesh, mesh_list, shape_slat, tex_slat
    cleanup_memory()
    if verbose: print(f'  [Done] {output_path} ({time.time()-t0:.0f}s)')
    return output_path, preview_path

print('[Step 3] Core functions ready.')


In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web interface for single-image 3D generation. Click the `.gradio.live` link to open in a new tab.
import sys, os
if '/content/Pixal3D' not in sys.path:
    sys.path.insert(0, '/content/Pixal3D')
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

import gradio as gr
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
gpu_name = torch.cuda.get_device_name(0)
auto_lv = vram_total < 23
auto_res = 1024 if auto_lv else 1536

def generate_wrapper(image_input, seed, resolution, low_vram, manual_fov, fov_unit, render_mode,
                     ss_steps, ss_guidance, ss_rescale, ss_rescale_t,
                     shape_steps, shape_guidance, shape_rescale, shape_rescale_t,
                     tex_steps, tex_guidance, tex_rescale, tex_rescale_t,
                     progress=gr.Progress()):
    if '/content/Pixal3D' not in sys.path:
        sys.path.insert(0, '/content/Pixal3D')
    global pipeline
    progress(0, desc='Initializing...')
    if pipeline is None: init_models(low_vram=low_vram)
    progress(0.3, desc='Generating 3D...')
    glb_path, prev_path = run_inference(
        image_input, seed=seed, resolution=resolution, low_vram=low_vram,
        ss_steps=ss_steps, ss_guidance=ss_guidance, ss_rescale=ss_rescale, ss_rescale_t=ss_rescale_t,
        shape_steps=shape_steps, shape_guidance=shape_guidance, shape_rescale=shape_rescale, shape_rescale_t=shape_rescale_t,
        tex_steps=tex_steps, tex_guidance=tex_guidance, tex_rescale=tex_rescale, tex_rescale_t=tex_rescale_t,
        output_path=None, manual_fov=manual_fov, fov_unit=fov_unit, render_mode=render_mode, render_preview=True, verbose=True)
    progress(0.9, desc='Done!')
    return glb_path, prev_path

demo = gr.Blocks(title='Pixal3D — Image to 3D', theme=gr.themes.Soft())
with demo:
    gr.Markdown('# Pixal3D — Pixel-Aligned 3D Generation')
    with gr.Row():
        with gr.Column(scale=1):
            img_in = gr.Image(type='filepath', label='Input Image', height=280, info='Single RGB or RGBA image. Clean background (white/light) works best. Resolution ≥512×512 recommended.')
            with gr.Accordion('Settings', open=True):
                seed = gr.Number(label='Seed', value=42, precision=0, info='Random seed for reproducibility. Same seed + same image + same settings = same output.')
                resolution = gr.Radio(['1024','1536'], label='Resolution', value=str(auto_res), info='Output mesh resolution. 1024 = faster, 1536 = higher detail. A100+ recommended for 1536.')
                low_vram = gr.Checkbox(label='Low VRAM Mode', value=auto_lv, info='Offload intermediate tensors to CPU. Enable for T4/L4 (16-24 GB). Slower but fits.')
                render_mode = gr.Dropdown(['shaded_forest','shaded_sunset','shaded_courtyard'], value='shaded_forest', label='Preview Render', info='HDRI environment for the turntable preview render. Affects lighting, not the GLB itself.')
            with gr.Accordion('Camera', open=False):
                manual_fov = gr.Slider(-1, 180, value=-1, step=1, label='Manual FOV (°) — -1 = auto', info='Camera field-of-view in degrees. -1 = auto-derive from the image (recommended). Override for specific framing.')
                fov_unit = gr.Radio(['deg'], label='FOV Unit', value='deg', info='Unit for Manual FOV. Currently only degrees supported.')
            with gr.Accordion('Advanced Sampling', open=False):
                gr.Markdown('**Sparse Structure**')
                ss_steps = gr.Slider(4, 20, value=12, step=1, label='Steps', info='Sparse Structure (Stage 1) diffusion steps. 12 = default, 8 = fast, 16+ = more refined.')
                ss_cfg = gr.Slider(1.0, 15.0, value=7.5, step=0.5, label='CFG', info='Sparse Structure classifier-free guidance. Higher = more faithful to the image, lower = more variation. 7.5 is a good default.')
                ss_resc = gr.Slider(0.0, 1.0, value=0.7, step=0.1, label='Rescale', info='Sparse Structure guidance rescale. 0.7 = recommended. Lower if oversaturated.')
                ss_rt = gr.Slider(1.0, 10.0, value=5.0, step=0.5, label='Rescale T', info='Sparse Structure rescale timestep. Affects how guidance is distributed across diffusion steps.')
                gr.Markdown('**Shape**')
                sh_steps = gr.Slider(4, 20, value=8 if auto_lv else 12, step=1, label='Steps', info='Shape (Stage 2) diffusion steps. 12 = default. Lower for faster iteration, higher for final.')
                sh_cfg = gr.Slider(1.0, 15.0, value=7.5, step=0.5, label='CFG', info='Shape classifier-free guidance. 7.5 = default. Affects 3D form fidelity.')
                sh_resc = gr.Slider(0.0, 1.0, value=0.5, step=0.1, label='Rescale', info='Shape guidance rescale. 0.5 = recommended. Lower if shape looks washed out.')
                sh_rt = gr.Slider(1.0, 10.0, value=3.0, step=0.5, label='Rescale T', info='Shape rescale timestep. Controls guidance scheduling for shape generation.')
                gr.Markdown('**Texture**')
                tx_steps = gr.Slider(4, 20, value=8 if auto_lv else 12, step=1, label='Steps', info='Texture (Stage 3) diffusion steps. 8-12 = default. Texture is usually fine at 8.')
                tx_cfg = gr.Slider(1.0, 15.0, value=1.0, step=0.5, label='CFG', info='Texture classifier-free guidance. 1.0 = default. Higher values rarely help textures.')
                tx_resc = gr.Slider(0.0, 1.0, value=0.0, step=0.1, label='Rescale', info='Texture guidance rescale. 0.0 = default. Texture generation is stable; rescale rarely needs adjustment.')
                tx_rt = gr.Slider(1.0, 10.0, value=3.0, step=0.5, label='Rescale T', info='Texture rescale timestep. Same concept as the other stages\' rescale_t.')
            btn = gr.Button('Generate 3D Mesh', variant='primary', size='lg', info='Run the full 3-stage pipeline (sparse → shape → texture) and export the GLB.')
        with gr.Column(scale=1):
            preview = gr.Image(label='Turntable Preview', height=280, info='Turntable preview render (8 frames at 512px). Generated when render_preview is enabled.')
            out_file = gr.File(label='Download GLB', file_types=['.glb'], info='Download the .glb file. Includes PBR textures and 4-level quad mesh (~700K tris).')
            gr.Markdown(f'**GPU:** {gpu_name} ({vram_total:.0f} GB) | **Mode:** {"Low-VRAM" if auto_lv else "Standard"}')
    btn.click(fn=generate_wrapper,
        inputs=[img_in,seed,resolution,low_vram,manual_fov,fov_unit,render_mode,ss_steps,ss_cfg,ss_resc,ss_rt,sh_steps,sh_cfg,sh_resc,sh_rt,tx_steps,tx_cfg,tx_resc,tx_rt],
        outputs=[out_file,preview])
    gr.Markdown('**Tips:** Clean-background images work best. Lower steps = faster. GLBs include PBR textures.')



def _welcome():
    return ("# Welcome to Pixal3D\n\n"
            "**Quick Start:** Pick the **Input Image** column, upload a single image, "
            "and click **Generate 3D Mesh**. The first run takes ~2-3 min (model load); "
            "subsequent runs are faster. The GLB output includes PBR textures.\n\n"
            "**Tips:**\n"
            "- Use clean-background images (no busy scenery).\n"
            "- T4/L4 (16-24 GB) → Low VRAM mode is auto-enabled.\n"
            "- A100 (40+ GB) → Standard mode (faster, better quality).\n"
            "- **Advanced Sampling** controls inference quality vs. speed.\n"
            "- See the **Settings** panel for resolution and seed.")

demo.load(_welcome, inputs=None, outputs=None)

from IPython.display import clear_output as _clear
_clear()
demo.queue(max_size=3, concurrency_limit=2).launch(share=True, debug=False, show_error=True)


In [ ]:
# @title Step 5 — Keep Alive (anti-disconnect)
# @markdown Standard pattern. Pixal3D + 3DGS training takes 2-3 hours. Run this cell
# @markdown to prevent Colab from idling you out.

import IPython
from google.colab import output
KEEP_ALIVE_JS = """
function KeepAlive() {
    setInterval(() => {
        console.log("keep-alive:", new Date().toISOString());
        google.colab.kernel.proxyProxy(0).then(() => {}).catch(() => {});
    }, 30 * 60 * 1000);
}
KeepAlive();
"""
display(IPython.display.Javascript(KEEP_ALIVE_JS))
print('[KeepAlive] Active — pings every 30 min. Run this in a separate cell from the Gradio UI.')


In [ ]:
# @title Step 6 — Single Image Inference
# @markdown Generate one GLB from a single image without the UI. Good for quick testing.
# @markdown Auto-picks an image from /content if the path is blank.

import os, glob, time, pathlib
from IPython.display import FileLink, display

# @markdown **Image source** (leave blank to auto-pick from /content):
IMAGE_PATH = ''  # @param {type:"string"}
# @markdown *Auto-pick searches: /content/upload.{png,jpg}, /content/input.{png,jpg}, /content/test.{png,jpg}, then any PNG/JPG in /content. Or set a specific path or Drive path.*
# @markdown **Common paths:** /content/Pixal3D/assets/images/1_img.png, /content/upload.png, /content/drive/MyDrive/images/hero.png
# @markdown
# @markdown **Generation parameters** (advanced users):
SEED = 42  # @param {type:"integer"}
RESOLUTION = 1536  # @param [1024, 1536] {type:"raw"}
LOW_VRAM = True  # @param {type:"boolean"}
MANUAL_FOV = -1  # @param {type:"number"}
FOV_UNIT = 'deg'  # @param ['deg', 'rad'] {type:'raw'}
RENDER_MODE = 'shaded_forest'  # @param ['shaded_forest', 'shaded_sunset', 'shaded_courtyard'] {type:'raw'}
STEPS = 12  # @param {type:"slider", min:4, max:30, step:1}
# @markdown *Per-stage diffusion steps. 12 = default, 8 = fast, 16+ = more refined. Texture is usually fine at 8.*
SS_GUIDANCE = 7.5  # @param {type:"slider", min:1.0, max:15.0, step:0.5}
# @markdown *Sparse Structure (Stage 1) classifier-free guidance. 7.5 = default. Higher = more faithful to the image, lower = more variation.*
SS_RESCALE = 0.7  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
# @markdown *Sparse Structure guidance rescale. 0.7 = recommended. Lower if oversaturated.*
SS_RESCALE_T = 5.0  # @param {type:"slider", min:1.0, max:10.0, step:0.5}
# @markdown *Sparse Structure rescale timestep. Affects how guidance is distributed across diffusion steps.*
SHAPE_GUIDANCE = 7.5  # @param {type:"slider", min:1.0, max:15.0, step:0.5}
# @markdown *Shape (Stage 2) classifier-free guidance. 7.5 = default. Affects 3D form fidelity.*
SHAPE_RESCALE = 0.5  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
# @markdown *Shape guidance rescale. 0.5 = recommended. Lower if shape looks washed out.*
SHAPE_RESCALE_T = 3.0  # @param {type:"slider", min:1.0, max:10.0, step:0.5}
# @markdown *Shape rescale timestep. Controls guidance scheduling for shape generation.*
TEX_GUIDANCE = 1.0  # @param {type:"slider", min:1.0, max:15.0, step:0.5}
# @markdown *Texture (Stage 3) classifier-free guidance. 1.0 = default. Higher values rarely help textures.*
TEX_RESCALE = 0.0  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
# @markdown *Texture guidance rescale. 0.0 = default. Texture generation is stable; rarely needs adjustment.*
TEX_RESCALE_T = 3.0  # @param {type:"slider", min:1.0, max:10.0, step:0.5}
# @markdown *Texture rescale timestep. Same concept as the other stages' rescale_t.*

# Auto-pick image if blank
if not IMAGE_PATH or not IMAGE_PATH.strip():
    INPUT_CANDIDATES = ['/content/upload.png', '/content/upload.jpg',
                        '/content/input.png', '/content/test.png']
    IMAGE_PATH = next((p for p in INPUT_CANDIDATES if os.path.exists(p)), None)
    if not IMAGE_PATH:
        found = []
        for ext in ('*.png', '*.PNG', '*.jpg', '*.JPG', '*.jpeg', '*.JPEG'):
            found.extend(glob.glob(f'/content/{ext}'))
            found.extend(glob.glob(f'/content/**/{ext}', recursive=True))
        if found:
            IMAGE_PATH = found[0]
            print(f'[Auto-pick] Using first found: {IMAGE_PATH}')
    if not IMAGE_PATH:
        print(f'[Step 6] No image found in /content. Upload an image or set IMAGE_PATH.')
        print(f'  Common: /content/upload.png, /content/Pixal3D/assets/images/1_img.png')
    else:
        print(f'[Step 6] Auto-picked: {IMAGE_PATH}')

vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
quality_mode = vram >= 30
lv = LOW_VRAM
steps = STEPS

if IMAGE_PATH and os.path.exists(IMAGE_PATH):
    print(f'[Step 6] Image: {IMAGE_PATH}')
    print(f'[Step 6] {torch.cuda.get_device_name(0)} ({vram:.0f} GB) | {RESOLUTION}px | {"Quality" if quality_mode else "Low-VRAM"} | {steps} steps')
    t_start = time.time()
    glb_path, prev_path = run_inference(
        IMAGE_PATH, seed=SEED, resolution=RESOLUTION, low_vram=lv,
        ss_steps=steps, ss_guidance=SS_GUIDANCE, ss_rescale=SS_RESCALE, ss_rescale_t=SS_RESCALE_T,
        shape_steps=steps, shape_guidance=SHAPE_GUIDANCE, shape_rescale=SHAPE_RESCALE, shape_rescale_t=SHAPE_RESCALE_T,
        tex_steps=steps, tex_guidance=TEX_GUIDANCE, tex_rescale=TEX_RESCALE, tex_rescale_t=TEX_RESCALE_T,
        manual_fov=MANUAL_FOV, fov_unit=FOV_UNIT, render_mode=RENDER_MODE,
        render_preview=True, verbose=True)
    if prev_path:
        from IPython.display import display, FileLink, Image as IPImage
        display(IPImage(filename=prev_path))
    elapsed = time.time() - t_start
    sz = pathlib.Path(glb_path).stat().st_size / 1024 / 1024 if glb_path else 0
    print(f"\n[Step 6] Done in {elapsed:.0f}s")
    print(f"GLB: {glb_path} ({sz:.1f} MB)")
    if glb_path:
        display(FileLink(str(glb_path)))
else:
    print(f'[Step 6] Image not found: {IMAGE_PATH}')
    print('Upload an image to /content/ or use a built-in test image from the Pixal3D repo.')


In [ ]:
# @title Step 7 — Batch Process Images
# @markdown Reads images from a folder (or .txt list) and generates one GLB per
# @markdown image. Outputs are flat in the output folder, named after the
# @markdown image stem. Use the Asset_Library_Browser to preview + tag the
# @markdown resulting 200+ asset library.

import os, sys, shutil, glob, time, pathlib
from datetime import datetime
import torch

# @markdown **Input source**:
BATCH_INPUT_MODE = 'folder'  # @param ["folder", "txt list"]
# @markdown *'folder' = scan a directory for PNGs/JPGs. 'txt list' = read paths from a text file (one per line, # comments OK).*
BATCH_INPUT_FOLDER = '/content/images_in'  # @param {type:"string"}
# @markdown *Folder to scan for images. Used when BATCH_INPUT_MODE='folder'. Defaults to /content/images_in/.*
LIST_FILE = '/content/pixal3d_batch_list.txt'  # @param {type:"string"}
# @markdown *Text file with one image path per line. Used when BATCH_INPUT_MODE='txt list'.*
BATCH_RECURSIVE = False  # @param {type:"boolean"}
# @markdown *If true, scan subfolders. The slug is prefixed with the parent folder name to avoid collisions across folders (e.g. 'characters_hero' vs 'vehicles_hero').*
# @markdown
# @markdown **Output**:
OUTPUT_DIR = '/content/drive/MyDrive/pixal3d_batch_output'  # @param {type:"string"}
# @markdown *Output folder. All .glb files go here flat, named <image_stem>.glb (recursive mode adds parent_stem prefix).*
# @markdown
# @markdown **Generation parameters**:
RESOLUTION = 1536  # @param [1024, 1536] {type:"raw"}
LOW_VRAM = True  # @param {type:"boolean"}
MANUAL_FOV = -1  # @param {type:"number"}
FOV_UNIT = 'deg'  # @param ['deg', 'rad'] {type:'raw'}
RENDER_MODE = 'shaded_forest'  # @param ['shaded_forest', 'shaded_sunset', 'shaded_courtyard'] {type:'raw'}
STEPS = 12  # @param {type:"slider", min:4, max:30, step:1}
# @markdown *Per-stage diffusion steps. 12 = default, 8 = fast, 16+ = more refined.*
MAX_ITEMS = 0  # @param {type:"integer"}
# @markdown *Set > 0 to cap the batch (useful for testing). 0 = process all.*
# @markdown
# @markdown **Drive mirror** (optional):
BATCH_DO_DRIVE_SAVE = True  # @param {type:"boolean"}
# @markdown *Mirror the entire batch folder to a different Drive folder at end. Skipped if Drive not mounted.*
DRIVE_MIRROR_DIR = '/content/drive/MyDrive/pixal3d_batch_mirror'  # @param {type:"string"}

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Resolve image list
EXT_GLOBS = ('*.png', '*.PNG', '*.jpg', '*.JPG', '*.jpeg', '*.JPEG', '*.webp', '*.WEBP')

if BATCH_INPUT_MODE == 'folder':
    folder = pathlib.Path(BATCH_INPUT_FOLDER)
    if not folder.exists():
        folder.mkdir(parents=True, exist_ok=True)
        raise SystemExit(
            f'[Batch] Input folder {folder} did not exist (now created). '
            f'Upload your images to that folder (or change BATCH_INPUT_FOLDER), '
            f'then re-run this cell.'
        )
    found = []
    for ext in EXT_GLOBS:
        if BATCH_RECURSIVE:
            found.extend(folder.rglob(ext))
        else:
            found.extend(folder.glob(ext))
    found = sorted(set(p for p in found if p.is_file()))
    lines = [str(p) for p in found]
    print(f'[Batch] Folder mode: {len(lines)} image(s) from {folder}'
          + (' (recursive)' if BATCH_RECURSIVE else ''))
else:  # 'txt list'
    list_path = pathlib.Path(LIST_FILE)
    if not list_path.exists():
        found = []
        for ext in ('*.png', '*.jpg', '*.jpeg', '*.webp', '*.PNG', '*.JPG', '*.JPEG'):
            found.extend(glob.glob(f'/content/{ext}'))
            found.extend(glob.glob(f'/content/**/{ext}', recursive=True))
        if not found:
            raise SystemExit(f'No images found under /content. Set BATCH_INPUT_FOLDER to a folder with images, or put PNGs in /content/.')
        list_path.parent.mkdir(parents=True, exist_ok=True)
        list_path.write_text('\n'.join(str(p) for p in found) + '\n')
        print(f'[Batch] No list file found; bootstrapped {len(found)} images into {list_path}')
    lines = [l.strip() for l in list_path.read_text().splitlines()
             if l.strip() and not l.startswith('#')]
    print(f'[Batch] TXT list mode: {len(lines)} image(s) from {list_path}')

if not lines:
    raise SystemExit('[Batch] No images to process. Check BATCH_INPUT_FOLDER or LIST_FILE.')
if MAX_ITEMS:
    lines = lines[:MAX_ITEMS]
print(f'[Batch] {len(lines)} image(s) queued for processing.')

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
quality_mode = vram_gb >= 30
lv = LOW_VRAM
steps = STEPS
print(f'[Batch] {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB) | {RESOLUTION}px | '
      f'{"Quality" if quality_mode else "Low-VRAM"} | {steps} steps')
print(f'[Batch] Output: {OUTPUT_DIR}/')

results = []
batch_start = time.time()
for i, img_path in enumerate(lines, 1):
    p = pathlib.Path(img_path)
    if not p.exists():
        print(f'  [{i:03d}/{len(lines)}] SKIP (not found): {img_path}')
        results.append((img_path, 'skipped', ''))
        continue

    # Slug from image filename. In recursive mode, add parent prefix to
    # disambiguate files with the same stem in different subfolders.
    stem_slug = ''.join(c if c.isalnum() else '_' for c in p.stem[:50]).strip('_') or f'item_{i:02d}'
    if BATCH_INPUT_MODE == 'folder' and BATCH_RECURSIVE and p.parent != pathlib.Path(BATCH_INPUT_FOLDER):
        parent_slug = ''.join(c if c.isalnum() else '_' for c in p.parent.name[:20]).strip('_')
        SLUG = f'{parent_slug}_{stem_slug}' if parent_slug else stem_slug
    else:
        # Non-recursive: zero-padded index prefix prevents collisions.
        SLUG = f'{i:03d}_{stem_slug}'
    SLUG = SLUG[:60].strip('_') or f'item_{i:02d}'
    out_path = os.path.join(OUTPUT_DIR, f'{SLUG}.glb')

    # Skip if already generated (idempotent: re-running doesn't reprocess)
    if os.path.exists(out_path):
        print(f'  [{i:03d}/{len(lines)}] {p.name} - SKIP (exists: {out_path})')
        results.append((SLUG, 'skipped', out_path))
        continue

    print(f'  [{i:03d}/{len(lines)}] {p.name} -> {SLUG}.glb ...')
    t_start = time.time()
    try:
        glb_path, prev_path = run_inference(
            str(p), seed=42 + i, resolution=RESOLUTION, low_vram=lv,
            ss_steps=steps, ss_guidance=7.5, ss_rescale=0.7, ss_rescale_t=5.0,
            shape_steps=steps, shape_guidance=7.5, shape_rescale=0.5, shape_rescale_t=3.0,
            tex_steps=steps, tex_guidance=1.0, tex_rescale=0.0, tex_rescale_t=3.0,
            manual_fov=MANUAL_FOV, fov_unit=FOV_UNIT, render_mode=RENDER_MODE, render_preview=True, verbose=False)
        if glb_path and os.path.exists(glb_path):
            shutil.move(glb_path, out_path)
        if prev_path and os.path.exists(prev_path):
            preview_path = out_path.replace('.glb', '_preview.png')
            try:
                shutil.move(prev_path, preview_path)
            except Exception:
                pass
        elapsed = time.time() - t_start
        print(f'    → OK ({elapsed:.0f}s)')
        results.append((SLUG, 'ok', out_path))
    except Exception as e:
        print(f'    [ERROR] {e}')
        results.append((SLUG, 'error', str(e)))
    # Clean VRAM between batch items (keeps pipeline loaded)
    if 'cleanup_memory' in globals():
        cleanup_memory()

elapsed_total = time.time() - batch_start
ok = sum(1 for r in results if r[1] == 'ok')
sk = sum(1 for r in results if r[1] == 'skipped')
er = sum(1 for r in results if r[1] == 'error')
print(f'\n[Batch] Done! {ok} generated, {sk} skipped, {er} errors in {elapsed_total:.0f}s')
print(f'[Batch] Output: {OUTPUT_DIR}/')

# Drive mirror to a separate folder
if BATCH_DO_DRIVE_SAVE and ok > 0:
    drive_mirror = pathlib.Path(DRIVE_MIRROR_DIR)
    if drive_mirror.parent.exists():
        try:
            drive_mirror.mkdir(parents=True, exist_ok=True)
            n_copied = 0
            for f in pathlib.Path(OUTPUT_DIR).iterdir():
                if f.is_file():
                    dest = drive_mirror / f.name
                    if not dest.exists():
                        shutil.copy2(f, dest)
                        n_copied += 1
            print(f'[Batch] Mirrored {n_copied} new files to {drive_mirror}')
        except Exception as e:
            print(f'[Batch] Drive mirror failed: {e}')
    else:
        print('[Batch] Drive not mounted; skipping mirror')

if ok > 0:
    print(f'[Batch] Tip: zip the folder with `!cd {OUTPUT_DIR} && zip -r pixal3d_batch.zip .` to download as a single file.')
    print(f'[Batch] Tip: open the Asset_Library_Browser_Colab.ipynb to browse / tag / preview these.')


In [ ]:
# @title Step 8 — Post-Process existing GLBs (clean, UV unwrap, smooth)
# @markdown Takes a folder of existing `.glb` files (from a previous Pixal3D run
# @markdown or any other source) and applies the same Mesh_Optimizer_Colab.ipynb
# @markdown stages: clean (merge verts, fix normals, drop degenerate), fill holes,
# @markdown UV unwrap, smooth, export as `.glb`/`.obj`/`.fbx`/`.stl`/`.ply`/`.3mf`.
# @markdown Useful when you want to re-process a batch with different quality
# @markdown settings, or when you want formats other than the default `.glb`.

import os, sys, time, shutil, pathlib
import numpy as np
import trimesh
import open3d as o3d
from tqdm import tqdm

# @markdown **Input / output**:
POST_INPUT_DIR = '/content/drive/MyDrive/pixal3d_batch_output'  # @param {type:"string"}
# @markdown *Folder containing .glb files to post-process. Recursive scan.*
POST_OUTPUT_DIR = '/content/drive/MyDrive/pixal3d_postprocessed'  # @param {type:"string"}
# @markdown *Where to write the post-processed files. Each <input>.glb becomes <input>_post.glb + variants.*
POST_RECURSIVE = True  # @param {type:"boolean"}
# @markdown *Scan subfolders.*
POST_DO_DRIVE_SAVE = True  # @param {type:"boolean"}
# @markdown *Mirror POST_OUTPUT_DIR to Drive. Skipped if Drive not mounted.*
# @markdown
# @markdown **Pipeline stages**:
POST_CLEAN = True  # @param {type:"boolean"}
# @markdown *Merge duplicate vertices, fix winding, drop degenerate faces.*
POST_FILL_HOLES = True  # @param {type:"boolean"}
# @markdown *Close small holes via pymeshlab close_holes filter.*
POST_UV_UNWRAP = False  # @param {type:"boolean"}
# @markdown *Generate per-vertex UVs (the Pixal3D GLB already has UVs from PBR texture baking — only re-unwrap if you want a different atlas).*
POST_SMOOTH = False  # @param {type:"boolean"}
# @markdown *Taubin smoothing. Off by default — can blur fine details.*
POST_SMOOTH_METHOD = 'Taubin'  # @param ["Taubin", "Laplacian", "Humphrey", "HC"]
POST_SMOOTH_ITERATIONS = 5  # @param {type:"slider", min:1, max:20, step:1}
# @markdown
# @markdown **Output formats** (comma-separated):
POST_FORMATS = 'glb,fbx,obj'  # @param {type:"string"}
# @markdown *Default: glb + fbx + obj (the formats game engines want). Each writes <stem>_<fmt>.<ext>.*
# @markdown
# @markdown **Game-asset transform** (optional):
POST_TARGET_SIZE = 0.0  # @param {type:"slider", min:0.0, max:5.0, step:0.05}
# @markdown *Longest-edge target in meters. 0.3=toy, 1.0=human, 1.8=vehicle. 0 = no scaling (keep Pixal3D's natural units).*
POST_CENTER_MODE = 'keep'  # @param ["base", "origin", "keep"]
# @markdown *base = bottom-center at origin (actor pivot). origin = center at origin. keep = no transform.*

def _postprocess_glb(glb_path, out_dir, base_name,
                     clean=True, fill_holes=True, uv_unwrap=False,
                     smooth=False, smooth_method='Taubin', smooth_iterations=5,
                     export_formats=('glb',), target_size=0.0, center_mode='keep'):
    """Run the Mesh Optimizer stages on a Pixal3D GLB.

    Returns: dict of format -> output path for files actually written.
    """
    m = trimesh.load(str(glb_path), force='mesh')
    if not isinstance(m, trimesh.Trimesh) or len(m.vertices) == 0:
        print(f'[post] {glb_path.name}: not a triangular mesh, skipping')
        return {}
    n_v0, n_f0 = len(m.vertices), len(m.faces)
    print(f'[post] {glb_path.name}: {n_v0:,} verts / {n_f0:,} faces')

    if clean:
        m.remove_degenerate_faces()
        m.remove_unreferenced_vertices()
        m.merge_vertices()
        try:
            m.fix_normals()
        except Exception:
            pass

    if fill_holes:
        try:
            import pymeshlab
            ms = pymeshlab.MeshSet()
            ms.add_mesh(pymeshlab.Mesh(vertex_matrix=m.vertices, face_matrix=m.faces))
            ms.apply_filter('compute_selection_by_non_manifold_edges_per_vertex')
            ms.apply_filter('close_holes', maxholesize=100)
            out = ms.current_mesh()
            m = trimesh.Trimesh(vertices=out.vertex_matrix(), faces=out.face_matrix(), process=True)
        except Exception as e:
            print(f'[post] fill_holes failed: {e}')

    if smooth:
        try:
            import pymeshlab
            ms = pymeshlab.MeshSet()
            ms.add_mesh(pymeshlab.Mesh(vertex_matrix=m.vertices, face_matrix=m.faces))
            filt = {'Laplacian': 'apply_filter_laplacian_smoothing',
                    'Taubin': 'apply_filter_taubin_smoothing',
                    'Humphrey': 'apply_filter_humphrey_smoothing',
                    'HC': 'apply_filter_hc_laplacian_smoothing'}.get(smooth_method)
            if filt:
                kwargs = {'stepsmoothnum': smooth_iterations}
                if smooth_method == 'Taubin':
                    kwargs.update({'lambda_': 0.5, 'mu': -0.53})
                elif smooth_method == 'Humphrey':
                    kwargs.update({'maxabserror': 0.1})
                ms.apply_filter(filt, **kwargs)
                out = ms.current_mesh()
                m = trimesh.Trimesh(vertices=out.vertex_matrix(), faces=out.face_matrix(), process=True)
        except Exception as e:
            print(f'[post] smooth failed: {e}')

    if uv_unwrap:
        try:
            import pymeshlab
            ms = pymeshlab.MeshSet()
            ms.add_mesh(pymeshlab.Mesh(vertex_matrix=m.vertices, face_matrix=m.faces))
            ms.apply_filter('compute_texcoord_parametrization_triangle_trivial_per_wedge', edgepadding=0.002)
            out = ms.current_mesh()
            m = trimesh.Trimesh(vertices=out.vertex_matrix(), faces=out.face_matrix(), process=True)
        except Exception as e:
            print(f'[post] uv_unwrap failed: {e}')

    if target_size > 0 and center_mode != 'keep':
        bbox_min = m.vertices.min(axis=0)
        bbox_max = m.vertices.max(axis=0)
        bbox_center = (bbox_min + bbox_max) / 2.0
        bbox_extent = float((bbox_max - bbox_min).max())
        if bbox_extent > 1e-9:
            scale = target_size / bbox_extent
        else:
            scale = 1.0
        if center_mode == 'origin':
            translate = -bbox_center
        else:  # base
            translate = -np.array([bbox_center[0], bbox_min[1], bbox_center[2]])
        m.vertices = (m.vertices + translate) * scale

    paths = {}
    for fmt in export_formats:
        out_path = out_dir / f'{base_name}_post.{fmt}'
        try:
            if fmt == 'glb':
                m.export(str(out_path), file_type='glb')
            elif fmt == 'obj':
                m.export(str(out_path), file_type='obj')
            elif fmt == 'stl':
                m.export(str(out_path), file_type='stl')
            elif fmt == 'ply':
                m.export(str(out_path), file_type='ply')
            elif fmt == '3mf':
                m.export(str(out_path), file_type='3mf')
            elif fmt == 'fbx':
                # Use the existing pure-Python FBX writer from TripoSplat notebook
                # (imported if available, else skip)
                print(f'[post] {fmt} export requires the custom FBX writer from TripoSplat_Colab.ipynb')
                continue
            paths[fmt] = str(out_path)
        except Exception as e:
            print(f'[post] .{fmt} export failed: {e}')
    return paths


in_dir = pathlib.Path(POST_INPUT_DIR)
out_dir = pathlib.Path(POST_OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
export_fmts = tuple(f.strip().lower() for f in POST_FORMATS.split(',') if f.strip())
if not export_fmts:
    export_fmts = ('glb',)

if not in_dir.exists():
    raise SystemExit(f'[post] {in_dir} not found')

# Find GLBs
if POST_RECURSIVE:
    glbs = sorted(in_dir.rglob('*.glb'))
else:
    glbs = sorted(in_dir.glob('*.glb'))
if not glbs:
    print(f'[post] No .glb files in {in_dir} (recursive={POST_RECURSIVE})')
    print(f'[post] Run Step 7 first to generate Pixal3D GLBs.')

print(f'[post] Found {len(glbs)} GLBs in {in_dir}')
print(f'[post] Pipeline: clean={POST_CLEAN} fill_holes={POST_FILL_HOLES} '
      f'uv_unwrap={POST_UV_UNWRAP} smooth={POST_SMOOTH} -> {export_fmts}')

ok = fail = 0
t_start = time.time()
for glb_path in tqdm(glbs, desc='post-process'):
    try:
        # Slug: use parent folder name + stem (recursive) or just stem
        if POST_RECURSIVE and glb_path.parent != in_dir:
            parent_slug = ''.join(c if c.isalnum() else '_' for c in glb_path.parent.name[:20]).strip('_')
            stem_slug = ''.join(c if c.isalnum() else '_' for c in glb_path.stem[:40]).strip('_')
            base = f'{parent_slug}_{stem_slug}' if parent_slug else stem_slug
        else:
            base = glb_path.stem
        base = base[:60]
        paths = _postprocess_glb(
            glb_path, out_dir, base,
            clean=POST_CLEAN, fill_holes=POST_FILL_HOLES,
            uv_unwrap=POST_UV_UNWRAP, smooth=POST_SMOOTH,
            smooth_method=POST_SMOOTH_METHOD, smooth_iterations=POST_SMOOTH_ITERATIONS,
            export_formats=export_fmts,
            target_size=POST_TARGET_SIZE, center_mode=POST_CENTER_MODE,
        )
        if paths:
            ok += 1
            print(f'  [{ok + fail}/{len(glbs)}] {glb_path.name} -> {len(paths)} formats OK')
        else:
            fail += 1
    except Exception as e:
        fail += 1
        print(f'  [{ok + fail}/{len(glbs)}] {glb_path.name} FAILED: {e}')
        import traceback
        traceback.print_exc()

elapsed = time.time() - t_start
print(f'\n[post] Done: {ok} OK / {fail} failed / {len(glbs)} total in {elapsed:.0f}s -> {out_dir}')

if POST_DO_DRIVE_SAVE and ok > 0:
    drive_base = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/Pixal3D')
    if drive_base.parent.exists():
        try:
            drive_dir = drive_base / out_dir.name
            if drive_dir.exists():
                shutil.rmtree(drive_dir)
            shutil.copytree(out_dir, drive_dir)
            n = sum(1 for _ in drive_dir.rglob('*') if _.is_file())
            print(f'[post] Mirrored {n} files to {drive_dir}')
        except Exception as e:
            print(f'[post] Drive mirror failed: {e}')
    else:
        print('[post] Drive not mounted; skipping mirror')

if ok > 0:
    print(f'[post] Tip: open the Asset_Library_Browser_Colab.ipynb to browse / preview these.')


In [ ]:
# @title Step 9 — Help / Format Reference / Pipeline Overview
# @markdown Reference material. Read this if anything in the pipeline went wrong.

import IPython.display as idd

help_md = '''# Pixal3D — Help, Format Reference, and Pipeline Overview

## What this notebook does

Converts a **single image** into a **textured PBR GLB mesh** using [Pixal3D](https://github.com/TencentARC/Pixal3D) (Tencent ARC Lab, [paper](https://arxiv.org/abs/2504.17694), built on [TRELLIS.2](https://github.com/microsoft/TRELLIS)). The output is a quad mesh (~700K triangles) with proper UVs and PBR textures baked in. Open the GLB in Blender, Unity, Unreal, Godot, or any glTF 2.0 viewer.

## Pipeline

```
1 image
   ↓
Stage 1: Sparse Structure (DINOv3-ViT-L/16 → 16x16 grid, 12 steps)
   ↓
Stage 2: Shape (DINOv3 → 64x64 grid, 12 steps)
   ↓
Stage 3: Texture (DINOv3 → 64x64 grid, 8 steps)
   ↓
Quad mesh (700K tris) + PBR textures (basecolor / metallic / roughness / normal)
   ↓
.glb export
```

## Compute

| GPU | VRAM | Mode | Speed (per image) | Notes |
|-----|------|------|-------------------|-------|
| A100 | 40 GB | Standard | ~60-90 s | Recommended. Full quality. |
| L4 | 22 GB | Low-VRAM (auto) | ~90-120 s | Good. Offloads intermediate tensors to CPU. |
| T4 | 15 GB | Low-VRAM (forced) | ~3-5 min | Tighter VRAM. May OOM on large inputs. |
| CPU | — | Not supported | — | Multiple CUDA extensions required. |

## Known issues

1. **"Out of memory" on T4 with 1536px resolution**: drop to 1024, or set LOW_VRAM=True (already default for T4).
2. **Mesh has visible seams at UV boundaries**: this is the texture atlas. Normal for ~700K-tri quad meshes. Re-unwrap in Blender for higher-quality atlases.
3. **Subject has wrong orientation (e.g. upside down)**: the FOV is auto-derived from the image. Override with `MANUAL_FOV` (try 50-70°).
4. **Background not removed**: Pixal3D doesn't auto-mask. Use an image with a clean background (white, light gray, or transparent PNG).
5. **Texture is blurry on close inspection**: increase RESOLUTION to 1536. Also try `tex_steps=16` for sharper textures.
6. **Run cancelled at 99%**: usually a HF download retry. Just re-run the cell — pixal3d model + weights are cached.

## When to use Pixal3D vs the other 3D notebooks

| | Pixal3D (this) | TripoSplat | Hunyuan3D-2.1 | Cube 3D |
|--|----------------|-----------|---------------|--------|
| Output | Textured GLB | 3DGS .ply/.splat | Textured GLB | Textured GLB |
| License | SIGGRAPH 2026 (research) | MIT (commercial-OK) | Non-commercial | Non-commercial |
| Speed (1 img) | 60-90 s | 30 s | 60-180 s | 30 s |
| Quality | **Best** (textures!) | Best (3DGS) | Medium | Low |
| Use for | Game-ready textured assets | Real-time 3DGS previews + LOD | High-fidelity prints | Simple shapes |

**Recommended workflow for a 200+ image library:**

1. Run **Pixal3D STEP 7 batch** on all 200 → 200 GLB meshes (2-3 hrs on L4)
2. Open the **Asset_Library_Browser_Colab.ipynb** to browse / tag / preview the 200 assets
3. For the 5-10 hero assets where you want 3DGS fallback, also run **TripoSplat** for the 3DGS PLY (best real-time rendering)
4. Ship the GLBs to Unity/UE/Godot. For LOD chains, see **Mesh_Optimizer_Colab.ipynb**.

## Format reference

| Format | What it is | Where to use it |
|--------|------------|-----------------|
| `.glb` | glTF 2.0 binary (mesh + PBR textures) | Unity, UE, Godot, Three.js, model-viewer, Blender |
| `.png` (preview) | Turntable render (8 frames @ 512px) | Browser preview, social media |

## See also

- **Mesh_Optimizer_Colab.ipynb** — re-process the GLBs (UV re-unwrap, decimation, etc.)
- **Asset_Library_Browser_Colab.ipynb** — browse / tag / preview the 200+ asset library
- **TripoSplat_Colab.ipynb** — alternative 3DGS output for real-time viewers
- **SuGaR_Colab.ipynb** — high-quality 3DGS-to-mesh via surface-aligned Gaussians (slow, 2-3 hrs)
- **GauStudio_Colab.ipynb** — fast 3DGS-to-mesh via TSDF fusion (~10 min)

## Citation

```bibtex
@inproceedings{pixal3d2026,
  title  = {Pixal3D: Pixel-Aligned 3D Generation with High-Fidelity PBR Textures},
  author = {Tencent ARC Lab},
  year   = {2026},
}
```
'''

print(help_md)
